In [ ]:
# BERT 모델 정의 셀 활성화
# Load a DistilBERT model.
import keras_nlp # Import keras_nlp
preset= "distil_bert_base_en_uncased"

# Use a shorter sequence length.
preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(preset,
                                                                   sequence_length=160,
                                                                   name="preprocessor_4_tweets"
                                                                  )

# Pretrained classifier.
classifier = keras_nlp.models.DistilBertClassifier.from_preset(preset,
                                                               preprocessor = preprocessor,
                                                               num_classes=2)

classifier.summary()

In [ ]:
X_train_mid = X_train_stratified.iloc[mid_train_indices_stratified]
y_train_mid = y_train_stratified.iloc[mid_train_indices_stratified]

X_test_mid = X_test_stratified.iloc[mid_test_indices_stratified]
y_test_mid = y_test_stratified.iloc[mid_test_indices_stratified]

In [ ]:
BATCH_SIZE = 32
EPOCHS = 2 # You can adjust the number of epochs as needed

# These variables are not strictly necessary for this setup but are kept for context
# NUM_TRAINING_EXAMPLES = X_train_stratified.shape[0]
# STEPS_PER_EPOCH = int(NUM_TRAINING_EXAMPLES) // BATCH_SIZE
# VALIDATION_DATA_SIZE = X_test_stratified.shape[0]

In [ ]:
# Compile
classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
    optimizer=keras.optimizers.Adam(1e-5),
    metrics= ["accuracy"]
)

# Fit

history = classifier.fit(x=X_train_mid,
                         y=y_train_mid,
                         batch_size=BATCH_SIZE,
                         epochs=EPOCHS, 
                         validation_data=(X_test_mid, y_test_mid) 
                        )

In [ ]:
def displayConfusionMatrix(y_true, y_pred, dataset):
   
    if len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
        y_pred_classes = np.argmax(y_pred, axis=1)
    else:
        y_pred_classes = y_pred


    total_samples = len(y_true)
    correct_predictions = np.sum(y_true == y_pred_classes)
    accuracy = correct_predictions / total_samples

    print(f"\n=== Performance on {dataset} Dataset ===")
    print(f"Total samples: {total_samples}")
    print(f"Correct predictions: {correct_predictions}")
    print(f"Accuracy: {accuracy:.4f}")


    cm = confusion_matrix(y_true, y_pred_classes)
    print("\nConfusion Matrix (Text):")
    print(cm)

   

In [ ]:
# 전체 테스트 데이터셋에 대해 예측 수행
y_pred_test = classifier.predict(X_test)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score

if len(y_pred_test.shape) > 1 and y_pred_test.shape[1] > 1:
    y_pred_test_classes = np.argmax(y_pred_test, axis=1)
else:
    y_pred_test_classes = y_pred_test 


print("\n=== Performance on Test Dataset ===")
total_samples = len(y_test)
correct_predictions = np.sum(y_test == y_pred_test_classes)
accuracy = accuracy_score(y_test, y_pred_test_classes)
precision = precision_score(y_test, y_pred_test_classes)
recall = recall_score(y_test, y_pred_test_classes)
f1 = f1_score(y_test, y_pred_test_classes)
cm = confusion_matrix(y_test, y_pred_test_classes)

print(f"Total samples: {total_samples}")
print(f"Correct predictions: {correct_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


print("\nConfusion Matrix (Text):")
print(cm)

# ROC-AUC score calculation requires probabilities, not class labels
# If the classifier has a predict_proba method, use it:
try:
    y_prob_test = classifier.predict(X_test) # Assuming predict returns logits
    roc_auc = roc_auc_score(y_test, y_prob_test[:, 1])
    print(f"ROC-AUC: {roc_auc:.4f}")
except AttributeError:
    print("ROC-AUC score could not be calculated as the model does not have predict_proba.")


# displayConfusionMatrix(y_test, y_pred_test, "Test") # This custom function is no longer needed here